In [1]:
# Data processing notebook that downloads and combines MASS floating-layer
# data files for a selected year (2015–present) using Linux `curl` and
# file concatenation commands -- Example:
#
# curl -s https://www.cfht.hawaii.edu/ObsInfo/Weather/mkam/MASSdata/out/ \
# | grep -o '18[^"]*\.mass\.bz2' \
# | while read f; do
#     curl -O "https://www.cfht.hawaii.edu/ObsInfo/Weather/mkam/MASSdata/out/$f"
#   done
#
# cat 18*.mass.bz2 > ../2018_floating.mass.bz2
#
# Individual `.mass.bz2` files are downloaded from the CFHT archive,
# merged into a single yearly MASS file, and decompressed into a `.mass`
# file for analysis.
#
# The notebook then parses the `.mass` file into a structured CSV using
# the `read_mass_file()` function, extracting floating- and fixed-layer
# seeing measurements, layer heights, and turbulence strengths.
#
# Timestamps are converted from UTC to HST before the finalized yearly
# dataset is written to CSV for downstream analysis.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def read_mass_file(filename):
    rows_by_time = {}

    with open(filename) as f:
        for line in f:
            line = line.strip()

            # Skip non-data lines
            if not line.startswith("T"):
                continue

            parts = line.split()
            timestamp = f"{parts[1]} {parts[2]}"
            kind = parts[3]

            # Get or create row for this timestamp
            if timestamp not in rows_by_time:
                rows_by_time[timestamp] = {"timestamp": timestamp}

            row = rows_by_time[timestamp]

            # Floating layers (L)
            if kind == "L":
                row["floating_seeing"] = float(parts[6])

                row["float_h1"] = float(parts[7])
                row["float_c1"] = float(parts[8])

                row["float_h2"] = float(parts[9])
                row["float_c2"] = float(parts[10])

                row["float_h3"] = float(parts[11])
                row["float_c3"] = float(parts[12])

            # Fixed layers (X)
            elif kind == "X":
                row["fixed_seeing"] = float(parts[6])

                heights = [float(parts[i]) for i in range(7, len(parts), 2)]
                strengths = [float(parts[i]) for i in range(8, len(parts), 2)]

                for i, (h, c) in enumerate(zip(heights, strengths), start=1):
                    row[f"fixed_h{i}"] = h
                    row[f"fixed_c{i}"] = c
    
    # Convert to DataFrame
    df = pd.DataFrame(rows_by_time.values())

    # Parse timestamp column
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Sort chronologically
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df

In [20]:
df = read_mass_file("./2025_floating.mass")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True) \
                    .dt.tz_convert("US/Hawaii") \
                    .dt.tz_localize(None)
df.to_csv("2025_floating.csv", index=False)
print('done')

## bzip2 -d 25*.mass.bz2
## then cat 25*.mass > ../2025_floating.mass

done


In [21]:
# Load both CSV files
floating = pd.read_csv("./MASS_files/2025_floating.csv")
sec90 = pd.read_csv("./years_tol90sec/2025_90sec.csv")

# Remove rows with any NaN values in the floating dataframe
floating = floating.dropna()

# Convert timestamp columns to datetime
floating['timestamp'] = pd.to_datetime(floating['timestamp'])
sec90['dimm_time'] = pd.to_datetime(sec90['dimm_time'])

# Sort both dataframes by their timestamp (required for merge_asof)
floating = floating.sort_values('timestamp')
sec90 = sec90.sort_values('dimm_time')

# Merge on nearest timestamp with 1 second tolerance
merged = pd.merge_asof(
    sec90, 
    floating, 
    left_on='dimm_time', 
    right_on='timestamp', 
    direction='nearest', 
    tolerance=pd.Timedelta(seconds=90)  # adjust if needed
)

# Create a new column with the time difference in seconds
merged['df_dimm_floating'] = (merged['dimm_time'] - merged['timestamp']).dt.total_seconds()

# Drop the duplicate timestamp column from floating
merged = merged.drop(columns=['timestamp'])

merged = merged.dropna()
merged = merged.drop_duplicates()

# Save the result as 2015_merged.csv
merged.to_csv('2025_merged.csv', index=False)

print("Nearest-merge complete! Saved as '2025_merged.csv'")

Nearest-merge complete! Saved as '2025_merged.csv'


In [7]:
pd.set_option('display.max_columns', None)  # None = show all columns
merged_df = pd.read_csv("./years_merged/2018_merged.csv")
print(merged_df.loc[merged_df['float_c1'] > 100])

                dimm_time          500m           1km           2km  \
5228  2018-04-23 21:33:05  1.900000e-14  7.110000e-23  9.090000e-18   
5229  2018-04-23 21:34:35  1.900000e-14  7.110000e-23  9.090000e-18   
5234  2018-04-23 21:43:35  8.010000e-22  7.980000e-22  1.700000e-21   
5235  2018-04-23 21:45:05  8.010000e-22  7.980000e-22  1.700000e-21   
5238  2018-04-23 21:51:04  3.970000e-14  1.190000e-22  7.940000e-16   
5239  2018-04-23 21:52:34  3.970000e-14  1.190000e-22  7.940000e-16   
5249  2018-04-23 22:16:35  3.590000e-14  2.050000e-22  1.620000e-22   

               4km           8km          16km  mass_val  dimm_val  \
5228  1.710000e-14  1.590000e-15  1.400000e-14      0.22      0.39   
5229  1.710000e-14  1.590000e-15  1.400000e-14      0.22      0.35   
5234  6.200000e-15  7.670000e-16  1.220000e-14      0.12      0.44   
5235  6.200000e-15  7.670000e-16  1.220000e-14      0.12      0.53   
5238  2.590000e-15  1.450000e-16  1.860000e-14      0.24      0.42   
5239  2.590

In [19]:
# Returns True if any duplicate rows exist
has_duplicates = merged_df.duplicated().any()
print("Any duplicate rows?", has_duplicates)

Any duplicate rows? False
